# 82514 · Sesión S19 — Filtro de partículas y localización de Monte Carlo

**Bloque 6** · lunes 23 de noviembre de 2026 · 2 h · IQS Universitat Ramon Llull

Partículas que se mueven, se pesan y se remuestrean: en el pasillo de S18 y en el almacén.

| | |
|---|---|
| **En clase** | Seis minutos y medio: las secciones 0, 2 y 3 sin deteneros, y las 4 a 6, que se ponen en común al acabar. Después, el S19b. |
| **En casa** | Sección 1 y ejercicios; soluciones al final. |
| **Entrega** | No se sube: el taller se pone en común en clase. |

Cada paso tiene tres celdas: **▶ Parámetros**, lo único que se cambia; **⚙ Cálculo**, que no hace falta tocar; y **✔ Resultado**, lo que sale.

---

In [ ]:
# @title ⚙ Instalación · ejecutad esta celda primero
# Ejecutar una sola vez. En Colab tarda unos segundos.
import importlib, subprocess, sys

def asegurar(mods):
    faltan = []
    for pip_name, import_name in mods:
        try:
            importlib.import_module(import_name)
        except ImportError:
            faltan.append(pip_name)
    if faltan:
        print('Instalando:', ' '.join(faltan))
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + faltan, check=False)
    else:
        print('Todo instalado ya.')

asegurar([('numpy', 'numpy'), ('scipy', 'scipy'), ('matplotlib', 'matplotlib')])

import time
import numpy as np, scipy, matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.ndimage import distance_transform_edt

FIJAS = {'numpy': '2.4.6', 'scipy': '1.16.3', 'matplotlib': '3.10.0'}   # las de Colab con las que se probó el bloque 6
EN_COLAB = 'google.colab' in sys.modules
np.set_printoptions(precision=3, suppress=True)
plt.rcParams['figure.figsize'] = (10, 3.6)
plt.rcParams['axes.grid'] = True
plt.rcParams['axes.axisbelow'] = True
# los colores de todo el cuaderno: la verdad (el robot, su ruta), en negro; lo estimado (la nube, la estimación), en
# azul; el error, en rojo; en las comparaciones, lo que falla en rojo, lo que funciona en verde y lo de en medio en
# ámbar; la medida del LiDAR, en ámbar; las referencias, en gris
IQS_AZUL, IQS_VERDE, GRIS, AMBAR, ROJO = '#1B2A80', '#1FA355', '#8A8FA8', '#B9770E', '#C0392B'
COMA = FuncFormatter(lambda v, p: f'{v:g}'.replace('.', ',').replace('-', '−'))   # ejes con coma decimal

def coma(x, n=2):
    """Un número con coma decimal y signo menos, para imprimir."""
    return f'{x:.{n}f}'.replace('.', ',').replace('-', '−')

def ejes(*axs):
    for ax in axs:
        ax.xaxis.set_major_formatter(COMA); ax.yaxis.set_major_formatter(COMA)

for nombre, mod in (('numpy', np), ('scipy', scipy), ('matplotlib', matplotlib)):
    print(nombre, mod.__version__, '' if mod.__version__ == FIJAS[nombre] else f'(probado con {FIJAS[nombre]})')
print('en Colab:', EN_COLAB)
if not np.__version__.startswith('2.4'):
    print('AVISO: este cuaderno se probó con numpy 2.4; las cifras con semilla pueden cambiar un poco.')

## 0. El almacén, el robot y el LiDAR

*En clase, sin deteneros · diapositiva 87*

El simulador del S18b, más el pasillo, el LiDAR de S6, el campo de verosimilitud y MCL.

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo · el simulador: ejecutadlo sin abrirlo
# El simulador: el de S18b, más el pasillo con partículas, el LiDAR, MCL y el EKF sin nombres. Sin detenerse.
import numpy as np
from scipy.ndimage import distance_transform_edt
from scipy.optimize import least_squares

D2R = np.pi / 180
RES, FILAS, COLS = 0.10, 90, 130            # celdas de 10 cm: 13 m en x, 9 m en y


def mapa_almacen():
    """La rejilla de ocupacion del almacen de S22 (True = ocupado). Fila = y/0,1; columna = x/0,1."""
    occ = np.zeros((FILAS, COLS), bool)
    occ[0, :] = occ[-1, :] = True              # paredes exteriores
    occ[:, 0] = occ[:, -1] = True
    occ[42:47, :] = True                       # el muro central, de y = 4,2 a 4,7 m
    occ[42:47, 8:28] = False                   #   la puerta ancha, de x = 0,8 a 2,8 m
    occ[42:47, 92:100] = False                 #   la puerta estrecha, de x = 9,2 a 10,0 m
    occ[12:36, 40:47] = True                   # estanterias de la nave de abajo
    occ[12:36, 64:71] = True
    occ[58:78, 34:41] = True                   # estanterias de la nave de arriba
    occ[58:78, 58:65] = True
    occ[58:78, 82:89] = True
    return occ


OCC = mapa_almacen()
MUELLE, PICKING = np.array([0.6, 0.6]), np.array([8.3, 8.2])
BALIZAS = np.array([[0.1, 2.0], [3.0, 0.1], [8.0, 0.1], [12.9, 2.0], [5.5, 4.15],      # nave de abajo
                    [0.1, 7.0], [4.7, 8.9], [10.5, 8.9], [12.9, 6.5], [7.0, 4.75]])    # nave de arriba
RUTA = np.array([[0.6, 0.6], [1.8, 0.6], [1.8, 5.2], [9.6, 5.2], [9.6, 0.6], [0.6, 0.6]])   # la ruta del inventario

V, HZ = 0.5, 10                              # 0,5 m/s; odometria a 10 Hz
DD = V / HZ                                  # 5 cm por paso
SIG_D, SIG_T = 0.06 / np.sqrt(20), 0.2 * D2R # por paso: 13,4 mm y 0,2° (6 cm y 0,9° por metro)
Q = np.diag([SIG_D**2, SIG_T**2])            # Q: la covarianza del ruido de proceso (la odometria)
SIG_R, SIG_PHI, ALCANCE = 0.05, 1.0 * D2R, 8.0
R = np.diag([SIG_R**2, SIG_PHI**2])          # R: la covarianza del ruido de medida (el sensor de balizas)
CADA = 10                                    # un barrido de balizas cada 10 pasos: una vez por segundo
K95 = 2.4477                                 # en el plano, la elipse de 2,45σ encierra el 95 %


def envuelve(a):
    """Un angulo en (−π, π]: sin esto, la innovacion del rumbo salta 360°."""
    return (a + np.pi) % (2 * np.pi) - np.pi


def paso(x, d):
    """El modelo de movimiento: la pose x = (x; y; θ) compuesta con la odometria d = (δ_d; δ_θ) en SE(2),
    como en S8: avanzar δ_d en la direccion del rumbo y despues girar δ_θ."""
    return np.array([x[0] + d[0] * np.cos(x[2]), x[1] + d[0] * np.sin(x[2]), envuelve(x[2] + d[1])])


def jac_paso(x, d):
    """Sus jacobianos: F_x respecto a la pose (3 x 3) y F_v respecto a la odometria (3 x 2)."""
    c, s = np.cos(x[2]), np.sin(x[2])
    Fx = np.array([[1.0, 0.0, -d[0] * s], [0.0, 1.0, d[0] * c], [0.0, 0.0, 1.0]])
    Fv = np.array([[c, 0.0], [s, 0.0], [0.0, 1.0]])
    return Fx, Fv


def h(x, b):
    """El modelo de medida: distancia y rumbo a la baliza b desde la pose x."""
    dx, dy = b[0] - x[0], b[1] - x[1]
    return np.array([np.hypot(dx, dy), envuelve(np.arctan2(dy, dx) - x[2])])


def jac_h(x, b):
    """Su jacobiano H_x respecto a la pose (2 x 3)."""
    dx, dy = b[0] - x[0], b[1] - x[1]
    r2 = dx * dx + dy * dy
    r = np.sqrt(r2)
    return np.array([[-dx / r, -dy / r, 0.0], [dy / r2, -dx / r2, -1.0]])


def libre_vista(a, b):
    """¿Ve el sensor la baliza b desde a? Sin celdas ocupadas en medio (la baliza esta en la pared)."""
    n = int(np.ceil(np.linalg.norm(b - a) / 0.02))
    p = a + np.linspace(0, 1, n)[:, None] * (b - a)
    cc = np.clip((p[:, 0] / RES).astype(int), 0, COLS - 1)
    ff = np.clip((p[:, 1] / RES).astype(int), 0, FILAS - 1)
    return not OCC[ff, cc][:-6].any()


def ruta_densa():
    """La ruta del inventario, en pasos de 5 cm."""
    pts = [RUTA[0]]
    for a, b in zip(RUTA[:-1], RUTA[1:]):
        n = int(round(np.linalg.norm(b - a) / DD))
        pts += [a + (b - a) * k / n for k in range(1, n + 1)]
    return np.array(pts)


def mandos(rd):
    """La odometria verdadera de cada paso: el avance y el giro que llevan de un punto de la ruta al siguiente."""
    d, th = [], 0.0
    for a, b in zip(rd[:-1], rd[1:]):
        thn = np.arctan2(b[1] - a[1], b[0] - a[0])
        d.append((np.linalg.norm(b - a), envuelve(thn - th)))
        th = thn
    return np.array(d)


RD = ruta_densa()
U = mandos(RD)                               # 544 pasos, 27,2 m
VERDAD = [np.array([RD[0, 0], RD[0, 1], 0.0])]
for d in U:
    VERDAD.append(paso(VERDAD[-1], d))
VERDAD = np.array(VERDAD)
VISIBLES = [[j for j, b in enumerate(BALIZAS) if np.linalg.norm(b - p[:2]) < ALCANCE and libre_vista(p[:2], b)]
            for p in VERDAD]


def simula(semilla=1):
    """Lo que leen los sensores en una pasada por la ruta: la odometria de cada paso y, una vez por segundo,
    la distancia y el rumbo de cada baliza visible (con su numero: la asociacion se supone conocida)."""
    rng = np.random.default_rng(semilla)
    odo = U + rng.normal(0, [SIG_D, SIG_T], size=U.shape)
    med = {}
    for k in range(CADA, len(VERDAD), CADA):
        med[k] = [(j, h(VERDAD[k], BALIZAS[j]) + rng.normal(0, [SIG_R, SIG_PHI])) for j in VISIBLES[k]]
    return odo, med
# =====================================================================================================
# S19 · 1. El pasillo de S18, con particulas
# =====================================================================================================
NC = 130                                     # el pasillo junto al muro, y = 5,2 m: 130 celdas de 10 cm
XC = (np.arange(NC) + 0.5) * RES             # el centro de cada celda
P_HUECO, P_FALSA = 0.8, 0.1                  # el detector de huecos de S18: p(«hueco» | puerta), p(«hueco» | pared)
AV, SIG_AV = 0.5, np.sqrt(0.2) * 0.1        # cada medida, 0,5 m mas adelante, con σ = 4,47 cm (el nucleo 0,1/0,8/0,1 de S18)


def hueco_en(x):
    """¿Esta x frente a un hueco del muro? La puerta ancha, de 0,8 a 2,8 m; la estrecha, de 9,2 a 10,0 m."""
    x = np.asarray(x)
    return ((x >= 0.8) & (x < 2.8)) | ((x >= 9.2) & (x < 10.0))


def verosimilitud_hueco(x, z):
    """p(z | x) del detector: 0,8 o 0,1 si dice «hueco»; 0,2 o 0,9 si dice «pared». Fuera del pasillo, 0."""
    p = np.where(hueco_en(x), P_HUECO if z else 1 - P_HUECO, P_FALSA if z else 1 - P_FALSA)
    return p * ((x >= 0) & (x < 13.0))


def sistematico(w, rng):
    """El remuestreo sistematico: un solo numero al azar, r, y M punteros equiespaciados (r + k)/M sobre la
    suma acumulada de los pesos. Devuelve el indice de la particula que copia cada puntero."""
    M = w.size
    punteros = (rng.random() + np.arange(M)) / M
    return np.minimum(np.searchsorted(np.cumsum(w), punteros), M - 1)


def recorre_pasillo(semilla=18, x0=0.95, n=19):
    """El recorrido del pasillo de S18 (cuaderno S18a, recorre()): el robot sale de x0 y avanza 0,5 m por medida;
    el detector dice «hueco» o «pared», con sus errores. Las mismas medidas que vio el histograma de S18."""
    rng = np.random.default_rng(semilla)
    X, Z, x = [], [], x0
    for _ in range(n):
        X.append(x)
        Z.append(bool(rng.random() < (P_HUECO if hueco_en(XC[int(x / RES)]) else P_FALSA)))
        x += AV
    return np.array(X), np.array(Z)


def mcl_pasillo(Z, M, semilla):
    """MCL en el pasillo: M particulas repartidas al azar; para cada medida, pesar y remuestrear, y despues
    mover 0,5 m con ruido. Devuelve, por medida, las particulas y sus pesos (antes de remuestrear)."""
    rng = np.random.default_rng(semilla)
    P = rng.uniform(0, 13.0, M)
    hist = []
    for z in Z:
        w = verosimilitud_hueco(P, z)                    # 2. pesar
        w = w / w.sum() if w.sum() > 0 else np.full(M, 1 / M)
        hist.append((P.copy(), w.copy()))
        P = P[sistematico(w, rng)]                       # 3. remuestrear
        P = P + AV + rng.normal(0, SIG_AV, M)            # 1. muestrear el movimiento
    return hist


def histograma_pasillo(Z):
    """El filtro de histograma de S18 con las mismas medidas: 130 celdas; avanzar 4, 5 o 6 celdas con 0,1, 0,8 y
    0,1, y lo que saldria del pasillo se pierde (cuaderno S18a, corregir() y predecir())."""
    bel = np.full(NC, 1 / NC)
    hist = []
    for z in Z:
        bel = bel * verosimilitud_hueco(XC, z)
        bel /= bel.sum()
        hist.append(bel.copy())
        nueva = np.zeros_like(bel)
        for desp, w in zip((4, 5, 6), (0.1, 0.8, 0.1)):
            nueva[desp:] += w * bel[:NC - desp]
        bel = nueva / nueva.sum()
    return hist


def masa_cerca(P, w, x, d=0.3):
    """La creencia a ±d de la x verdadera: la suma de los pesos de las particulas que caen ahi."""
    return float(np.sum(w[np.abs(P - x) <= d + 1e-9]))
# =====================================================================================================
# S19 · 2. El LiDAR de S6 en la rejilla, el campo de verosimilitud y MCL en el almacen
# =====================================================================================================
ANG = np.radians(np.arange(0, 360, 0.25))    # 1440 haces: 360° con 0,25° de resolucion
RMAX, SIG_L, P_SIN = 10.0, 0.02, 0.02        # alcance de 10 m, 2 cm de ruido, 2 % de rayos sin retorno
HACES = np.arange(0, ANG.size, 80)           # el filtro usa 18 haces, uno cada 20°
DIST = distance_transform_edt(~OCC) * RES    # el campo de verosimilitud: distancia de cada celda al obstaculo
LIBRE = DIST > 0.28                          # donde cabe el centro del robot (radio de 0,28 m, el de S22)
_FL, _CL = np.nonzero(LIBRE)


def rayos(pose, angs, occ=OCC):
    """La distancia al primer obstaculo de cada rayo, marchando a pasos de 1 cm por la rejilla."""
    t = np.arange(0.01, RMAX, 0.01)
    a = pose[2] + angs
    cc = ((pose[0] + np.outer(np.cos(a), t)) / RES).astype(int)
    ff = ((pose[1] + np.outer(np.sin(a), t)) / RES).astype(int)
    fuera = (cc < 0) | (cc >= COLS) | (ff < 0) | (ff >= FILAS)
    choca = fuera | occ[np.clip(ff, 0, FILAS - 1), np.clip(cc, 0, COLS - 1)]
    r = t[choca.argmax(axis=1)]
    r[~choca.any(axis=1)] = RMAX
    return r


def barrido(pose, rng, idx=None):
    """Un barrido del LiDAR desde la pose (los haces idx; todos si no se dan), con su ruido y sus rayos perdidos."""
    a = ANG if idx is None else ANG[idx]
    r = rayos(pose, a) + rng.normal(0, SIG_L, a.size)
    r[rng.random(a.size) < P_SIN] = RMAX
    return np.minimum(r, RMAX)


def relativa(a, b):
    """La pose de b vista desde a, en SE(2) (la ⊖ de S8): adelante, a la izquierda y el giro."""
    c, s = np.cos(a[2]), np.sin(a[2])
    d = np.asarray(b[:2]) - np.asarray(a[:2])
    return np.array([c * d[0] + s * d[1], -s * d[0] + c * d[1], envuelve(b[2] - a[2])])
def log_verosimilitud(P, r, sig=0.3, zimp=0.5, zaz=0.5):
    """El campo de verosimilitud (Thrun et al., 2005): cada haz lleva su punto final a la rejilla desde cada
    particula, y vale una campana de σ = 0,3 m sobre la distancia de ese punto a la pared mas cercana, mas un
    fondo al azar. Suma de logaritmos sobre los 18 haces; una particula dentro de una pared vale −∞."""
    a, rr = ANG[HACES], r
    ok = rr < RMAX - 1e-6                                    # los rayos sin retorno no cuentan
    a, rr = a[ok], rr[ok]
    cc = ((P[:, 0:1] + rr * np.cos(P[:, 2:3] + a)) / RES).astype(int)
    ff = ((P[:, 1:2] + rr * np.sin(P[:, 2:3] + a)) / RES).astype(int)
    dentro = (cc >= 0) & (cc < COLS) & (ff >= 0) & (ff < FILAS)
    d = np.where(dentro, DIST[np.clip(ff, 0, FILAS - 1), np.clip(cc, 0, COLS - 1)], 2.0)
    p = zimp * np.exp(-0.5 * (d / sig) ** 2) / (np.sqrt(2 * np.pi) * sig) + zaz / RMAX
    lw = np.log(p).sum(axis=1)
    c0, f0 = (P[:, 0] / RES).astype(int), (P[:, 1] / RES).astype(int)
    en = (c0 >= 0) & (c0 < COLS) & (f0 >= 0) & (f0 < FILAS)
    lw[~en] = -np.inf
    lw[en] = np.where(LIBRE[f0[en], c0[en]], lw[en], -np.inf)
    return lw


def pesos(lw):
    """De logaritmos a pesos normalizados, sin desbordar. Si todas valen −∞, pesos iguales."""
    if not np.isfinite(lw).any():
        return np.full(lw.size, 1 / lw.size)
    w = np.exp(lw - lw[np.isfinite(lw)].max())
    return w / w.sum()


def al_azar(M, rng):
    """M particulas repartidas al azar por el sitio libre del almacen, con rumbos al azar."""
    i = rng.integers(0, _FL.size, M)
    return np.column_stack([(_CL[i] + rng.random(M)) * RES, (_FL[i] + rng.random(M)) * RES,
                            rng.uniform(-np.pi, np.pi, M)])


def alrededor(pose, M, rng, s=0.10, sth=3 * D2R):
    """M particulas alrededor de una pose conocida (el seguimiento): 10 cm y 3° de σ."""
    return np.column_stack([pose[0] + rng.normal(0, s, M), pose[1] + rng.normal(0, s, M),
                            envuelve(pose[2] + rng.normal(0, sth, M))])


def mover(P, d, rng, k=2.0):
    """Muestrear el movimiento: cada particula compone la pose relativa d que mide la odometria entre dos
    actualizaciones (0,5 m) con su propio ruido, k veces el de la odometria."""
    s = k * SIG_D * np.sqrt(10)
    ax, ay = d[0] + rng.normal(0, s, len(P)), d[1] + rng.normal(0, s, len(P))
    dt = d[2] + rng.normal(0, k * SIG_T * np.sqrt(10), len(P))
    c, sn = np.cos(P[:, 2]), np.sin(P[:, 2])
    return np.column_stack([P[:, 0] + ax * c - ay * sn, P[:, 1] + ax * sn + ay * c, envuelve(P[:, 2] + dt)])


def media_circular(a, w):
    """El rumbo medio de la nube: atan2(Σ w·sen θ, Σ w·cos θ). La media de los numeros no vale cerca de ±180°."""
    return np.arctan2(np.sum(w * np.sin(a)), np.sum(w * np.cos(a)))


def mcl_almacen(k0, n, M, semilla, P0=None, secuestro=None, inyecta=0.0, sensor='lidar'):
    """MCL en el almacen: el robot recorre la ruta desde el paso k0 y el filtro se actualiza cada 10 pasos (0,5 m),
    n veces. Las particulas se mueven con la odometria que mide el robot (con su error de 6 cm y 0,9° por metro).
    secuestro = (u, k_nuevo): en la actualizacion u, el robot aparece en el paso k_nuevo de la ruta. inyecta:
    fraccion de particulas que se cambian por otras al azar en cada actualizacion. sensor: 'lidar' (18 haces y el
    campo de verosimilitud) o 'balizas' (las diez balizas, sin decir cual es cada una: como el EKF de la seccion 5).
    Devuelve, por actualizacion, la verdad, la nube, los pesos, la estimacion y su error."""
    rng = np.random.default_rng(semilla)
    odo = U + np.random.default_rng(semilla + 1_000_000).normal(0, [SIG_D, SIG_T], size=U.shape)   # lo que mide el robot
    P = al_azar(M, rng) if P0 is None else P0.copy()
    k, out = k0, []
    for u in range(n + 1):
        if secuestro and u == secuestro[0]:
            k = secuestro[1]
        verdad = VERDAD[k]
        if sensor == 'lidar':                                                   # 2. pesar
            lw = log_verosimilitud(P, barrido(verdad, rng, HACES))
        else:
            zs = [h(verdad, BALIZAS[j]) + rng.normal(0, [SIG_R, SIG_PHI]) for j in VISIBLES[k]]
            lw = log_verosimilitud_balizas(P, zs)
        w = pesos(lw)
        est = np.array([np.sum(w * P[:, 0]), np.sum(w * P[:, 1]), media_circular(P[:, 2], w)])
        out.append(dict(k=k, verdad=verdad, P=P.copy(), w=w, est=est,
                        err=float(np.hypot(*(est[:2] - verdad[:2]))),
                        err_th=float(abs(envuelve(est[2] - verdad[2]))),
                        cerca=float(np.sum(w * (np.hypot(P[:, 0] - verdad[0], P[:, 1] - verdad[1]) < 0.5)))))
        if u == n:
            break
        P = P[sistematico(w, rng)]                                            # 3. remuestrear
        if inyecta > 0:
            m = int(round(inyecta * M))
            P[rng.choice(M, m, replace=False)] = al_azar(m, rng)
        k2 = min(k + CADA, len(U))
        d = np.zeros(3)
        for dd in odo[k:k2]:
            d = paso(d, dd)                     # la pose relativa que da la odometria entre dos actualizaciones
        P = mover(P, d, rng)                                                  # 1. muestrear el movimiento
        k = k2
    return out


def localizado(e, dist=0.3, ang=5 * D2R):
    """Exito de la localizacion: la estimacion a menos de 30 cm y 5° de la verdad."""
    return e["err"] < dist and e["err_th"] < ang
# =====================================================================================================
# S19 · 3. Cuando los reflectores no se distinguen: el EKF con el vecino mas cercano
# =====================================================================================================
PUERTA_99 = 9.21                             # χ² con 2 grados de libertad: el 99 % de las medidas buenas, por debajo


def log_verosimilitud_balizas(P, zs):
    """Balizas que no dicen cual son: cada medida (distancia y rumbo) pesa una mezcla sobre las diez balizas
    (1/10 cada una). Suma de logaritmos sobre las medidas; una particula dentro de una pared vale −∞."""
    lw = np.zeros(len(P))
    for z in zs:
        acc = np.zeros(len(P))
        for b in BALIZAS:
            dx, dy = b[0] - P[:, 0], b[1] - P[:, 1]
            er = z[0] - np.hypot(dx, dy)
            ep = envuelve(z[1] - envuelve(np.arctan2(dy, dx) - P[:, 2]))
            acc += np.exp(-0.5 * ((er / SIG_R) ** 2 + (ep / SIG_PHI) ** 2))
        lw += np.log(acc / len(BALIZAS) + 1e-12)
    c0, f0 = (P[:, 0] / RES).astype(int), (P[:, 1] / RES).astype(int)
    en = (c0 >= 0) & (c0 < COLS) & (f0 >= 0) & (f0 < FILAS)
    lw[~en] = -np.inf
    lw[en] = np.where(LIBRE[f0[en], c0[en]], lw[en], -np.inf)
    return lw


def simula_sin_nombre(semilla=1, k0=0):
    """Como simula(), desde el paso k0, pero el sensor no dice que baliza es: solo la distancia y el rumbo."""
    rng = np.random.default_rng(semilla)
    odo = U[k0:] + rng.normal(0, [SIG_D, SIG_T], size=U[k0:].shape)
    med = {}
    for k in range(k0 + CADA, len(VERDAD), CADA):
        med[k] = [h(VERDAD[k], BALIZAS[j]) + rng.normal(0, [SIG_R, SIG_PHI]) for j in VISIBLES[k]]
    return odo, med


def ekf_vecino(odo, med, x0, Sigma0, k0=0, puerta=PUERTA_99):
    """El EKF de S18 cuando no se sabe que baliza es: cada medida se atribuye a la baliza del mapa con menor
    d² = νᵀ·S⁻¹·ν desde la estimacion (el vecino mas cercano), y se descarta si pasa de la puerta."""
    x, S_ = np.array(x0, float), Sigma0.copy()
    X, P = [x.copy()], [S_.copy()]
    for k, d in enumerate(odo, start=k0 + 1):
        Fx, Fv = jac_paso(x, d)
        x = paso(x, d)
        S_ = Fx @ S_ @ Fx.T + Fv @ Q @ Fv.T
        for z in med.get(k, []):
            mejor = None
            for b in BALIZAS:
                Hx = jac_h(x, b)
                nu = z - h(x, b); nu[1] = envuelve(nu[1])
                Sm = Hx @ S_ @ Hx.T + R
                d2 = float(nu @ np.linalg.solve(Sm, nu))
                if mejor is None or d2 < mejor[0]:
                    mejor = (d2, Hx, nu, Sm)
            d2, Hx, nu, Sm = mejor
            if puerta is not None and d2 > puerta:
                continue
            L = S_ @ Hx.T @ np.linalg.inv(Sm)
            x = x + L @ nu; x[2] = envuelve(x[2])
            S_ = (np.eye(3) - L @ Hx) @ S_
        X.append(x.copy()); P.append(S_.copy())
    return np.array(X), np.array(P)

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
MAPA = ListedColormap(['white', '#4A4F63'])        # para las figuras: libre, en blanco; ocupado, en gris oscuro


def dibuja_almacen(ax, alfa=1.0, metros=False):
    """Para las figuras: el almacén de S22 (13 x 9 m); con los ejes en metros si metros."""
    ax.imshow(OCC, cmap=MAPA, origin='lower', extent=(0, 13, 0, 9), interpolation='nearest', alpha=alfa)
    ax.set_aspect('equal'); ax.grid(False)
    if metros:
        ax.set_xlim(-0.4, 13.4); ax.set_ylim(-0.4, 9.4); ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)'); ejes(ax)
    else:
        ax.set_xlim(0, 13); ax.set_ylim(0, 9); ax.set_xticks([]); ax.set_yticks([])


def dibuja_robot(ax, pose, color='k', z=6):
    """Para las figuras: el robot (0,28 m de radio) con una flecha en su rumbo. El de verdad, en negro."""
    ax.add_patch(plt.Circle(pose[:2], 0.28, fc='none', ec=color, lw=2, zorder=z))
    ax.arrow(pose[0], pose[1], 0.5 * np.cos(pose[2]), 0.5 * np.sin(pose[2]), width=0.05, head_width=0.22,
             head_length=0.18, fc=color, ec=color, zorder=z + 1, length_includes_head=True)


POSE_0 = VERDAD[54]                                         # el robot en el paso 54 de la ruta, en la nave de abajo
BARRIDO_0 = barrido(POSE_0, np.random.default_rng(0))       # un barrido del LiDAR: 1440 distancias, con su ruido
VUELVE_0 = BARRIDO_0 < RMAX - 1e-6                          # los haces que vuelven (los demás, sin retorno)
FIN_0 = POSE_0[:2] + BARRIDO_0[:, None] * np.column_stack([np.cos(POSE_0[2] + ANG), np.sin(POSE_0[2] + ANG)])

In [ ]:
# @title ✔ Resultado
print(f'Almacén de {COLS*RES:.0f} x {FILAS*RES:.0f} m ; sitio libre para el centro del robot: {LIBRE.sum()} celdas ; ruta de {len(U)} pasos')

fig, ax = plt.subplots(figsize=(9, 6.9))
dibuja_almacen(ax, metros=True)
ax.plot(VERDAD[:, 0], VERDAD[:, 1], color='k', lw=1.2, alpha=0.5, label='la ruta del inventario')
ax.plot(BALIZAS[:, 0], BALIZAS[:, 1], '*', color=IQS_VERDE, ms=14, mec='white', mew=0.6, zorder=5, label='las diez balizas')
ax.plot(FIN_0[VUELVE_0, 0], FIN_0[VUELVE_0, 1], '.', color=AMBAR, ms=2.5, zorder=4,
        label=f'un barrido del LiDAR: vuelven {VUELVE_0.sum()} de {ANG.size} haces')
for i in HACES[VUELVE_0[HACES]]:
    ax.plot([POSE_0[0], FIN_0[i, 0]], [POSE_0[1], FIN_0[i, 1]], color=AMBAR, lw=0.9, zorder=3)
ax.plot([], [], color=AMBAR, lw=0.9, label=f'los {HACES.size} haces del filtro, uno cada 20°')
dibuja_robot(ax, POSE_0)
ax.plot([], [], 'o', mfc='white', mec='k', mew=2, ms=10, label='el robot, en el paso 54')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=3, fontsize=9, frameon=False)
ax.set_title('El almacén de S22, la ruta y un barrido del LiDAR', fontsize=11)
plt.show()

**Qué tiene que salir:** 8100 celdas libres y 544 pasos.

## 1. El pasillo de S18 y las cinco partículas, paso a paso (en casa)

*En casa · diapositivas 71 a 86*

El filtro de Bayes de S18 con muestras: **muestrear** con la odometría, **pesar** con p(z | x) y **remuestrear** según el peso.

### Las cinco partículas de la diapositiva 72

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# las cinco partículas antes de moverse [m] y el ruido de cada una al avanzar 0,5 m [m]
CINCO = [1.2, 1.9, 4.0, 9.1, 11.0]  # @param {type:"raw"}
RUIDO_CINCO = [0.03, -0.02, 0.04, 0.01, -0.03]  # @param {type:"raw"}
# lo que dice el detector: True, «hueco»; False, «pared»
HUECO_CINCO = True  # @param {type:"boolean"}
# el número al azar del sorteo sistemático [de 0 a 1]
R_SORTEO = 0.5  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
C5_ANTES = np.array(CINCO)
C5 = C5_ANTES + AV + np.array(RUIDO_CINCO)           # 1. muestrear: avanzan 0,5 m, cada una con su ruido
C5_P = verosimilitud_hueco(C5, HUECO_CINCO)           # 2. pesar: p(z | x) de lo que dice el detector...
C5_W = C5_P / C5_P.sum()                              #    ... normalizada
C5_ACUM = np.cumsum(C5_W)                             # 3. remuestrear: la suma acumulada de los pesos,
C5_PUNTEROS = (R_SORTEO + np.arange(C5.size)) / C5.size   #    los M punteros (r + k)/M
C5_COPIA = np.searchsorted(C5_ACUM, C5_PUNTEROS)      #    y la partícula que copia cada uno
C5_COPIAS, C5_VECES = np.unique(C5_COPIA, return_counts=True)   # para la figura: cuántas copias de cada una

In [ ]:
# @title ✔ Resultado
print('tras muestrear:', '; '.join(coma(v) for v in C5))
print('p(z | x):', '; '.join(coma(v, 1) for v in C5_P), ' suma', coma(C5_P.sum(), 1))
print('pesos:', '; '.join(coma(v, 3) for v in C5_W))
print('acumulada:', '; '.join(coma(v, 3) for v in C5_ACUM), ' punteros:', '; '.join(coma(v, 1) for v in C5_PUNTEROS))
print('copian la', ', '.join(str(i + 1) for i in C5_COPIA), '->', '; '.join(coma(v) for v in C5[C5_COPIA]))

fig, ax = plt.subplots(figsize=(9, 4.1))
for a, b in ((0.8, 2.8), (9.2, 10.0)):
    ax.axvspan(a, b, color=IQS_VERDE, alpha=0.13, lw=0)
ax.text(1.8, 3.45, 'puerta ancha', ha='center', fontsize=9.5, color=IQS_VERDE)
ax.text(9.6, 3.45, 'estrecha', ha='center', fontsize=9.5, color=IQS_VERDE)
ax.scatter(C5_ANTES, np.full(C5.size, 3), s=90, color=IQS_AZUL, zorder=3)
ax.scatter(C5, np.full(C5.size, 2), s=90, color=IQS_AZUL, zorder=3)
for a, b in zip(C5_ANTES, C5):
    ax.annotate('', (b, 2.13), (a, 2.87), arrowprops=dict(arrowstyle='->', color=GRIS, lw=1.2))
ax.scatter(C5, np.full(C5.size, 1), s=900 * C5_W + 20, color=IQS_AZUL, alpha=0.85, zorder=3)
for i, (v, w_) in enumerate(zip(C5, C5_W)):
    dx = -0.35 if i == 0 and C5[1] - C5[0] < 1 else 0.35 if i == 1 and C5[1] - C5[0] < 1 else 0
    ax.text(v + dx, 0.5, coma(w_, 2), ha='center', fontsize=9.5, color='#333')
ax.text(12.9, 1.3, f"medida: «{'hueco' if HUECO_CINCO else 'pared'}»", ha='right', fontsize=9.5, color=AMBAR)
for j, n in zip(C5_COPIAS, C5_VECES):
    ax.scatter(np.full(n, C5[j]), -0.1 + 0.2 * np.arange(n), s=90, color=IQS_AZUL, zorder=3)
ax.set_yticks([3, 2, 1, 0], ['antes', '1 · muestrear', '2 · pesar', '3 · remuestrear'])
ax.set_ylim(-0.45, 3.75); ax.set_xlim(0, 13); ax.grid(False); ax.grid(axis='y', alpha=0.4); ax.xaxis.set_major_formatter(COMA)
ax.set_xlabel('x en el pasillo (m)'); ax.tick_params(axis='y', labelcolor=IQS_AZUL, length=0)
ax.set_title(f'Cinco partículas: el tamaño, su peso; copian la {", ".join(str(i + 1) for i in C5_COPIA)}', fontsize=11)
plt.show()

**Qué tiene que salir:** pesos 0,308 y 0,038; copian la 1, 1, 2, 4 y 4.

### MCL (la localización de Monte Carlo) en el pasillo de S18

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
X_PAS, Z_PAS = recorre_pasillo()                      # las medidas de S18
H_PAS = histograma_pasillo(Z_PAS)                       # el histograma de S18, con ellas
PF_PAS = mcl_pasillo(Z_PAS, 200, 1)                     # 200 partículas: en cada medida, la nube y sus pesos
# para la figura: cuatro medidas, la nube de cada una en las 130 celdas del pasillo, y en cada medida la creencia a
# ±0,3 m de la verdad, la de la nube y la del histograma
INST_PAS = (0, 3, 14, 18)
BORDES_PAS = np.arange(NC + 1) * RES
BARRAS_PAS = [np.histogram(PF_PAS[i][0], bins=NC, range=(0, 13), weights=PF_PAS[i][1])[0] for i in INST_PAS]
CERCA_PF = np.array([masa_cerca(P_, w_, x_) for (P_, w_), x_ in zip(PF_PAS, X_PAS)])
CERCA_H = np.array([h_[np.abs(XC - x_) <= 0.3 + 1e-9].sum() for h_, x_ in zip(H_PAS, X_PAS)])

In [ ]:
# @title ✔ Resultado
print('medidas:', ' '.join('H' if z else 'p' for z in Z_PAS))

fig, axs = plt.subplots(5, 1, figsize=(10, 8.6), sharex=True, gridspec_kw=dict(height_ratios=[1, 1, 1, 1, 1.7], hspace=0.55))
for ax, i, hh in zip(axs, INST_PAS, BARRAS_PAS):
    for a, b in ((0.8, 2.8), (9.2, 10.0)):
        ax.axvspan(a, b, color=IQS_VERDE, alpha=0.13, lw=0)
    ax.stairs(hh, BORDES_PAS, fill=True, color=IQS_AZUL, alpha=0.85, label='200 partículas')
    ax.stairs(H_PAS[i], BORDES_PAS, color=GRIS, lw=1.6, label='el histograma de S18')
    ax.axvline(X_PAS[i], color='k', ls='--', lw=1.4, label='la x verdadera')
    ax.set_yticks([]); ax.grid(False); ax.set_ylim(0, max(hh.max(), H_PAS[i].max()) * 1.15)
    ax.set_title(f"x = {coma(X_PAS[i])} m, «{'hueco' if Z_PAS[i] else 'pared'}»: a ±0,3 m, {coma(100*CERCA_PF[i], 0)} %"
                 f" (histograma: {coma(100*CERCA_H[i], 0)} %)", fontsize=10, loc='left')
axs[0].legend(handles=axs[0].get_legend_handles_labels()[0] + [Patch(color=IQS_VERDE, alpha=0.13, label='los huecos del muro')],
              fontsize=9, ncol=4, loc='lower right', bbox_to_anchor=(1, 1.12), frameon=False)
ax = axs[-1]
for a, b in ((0.8, 2.8), (9.2, 10.0)):
    ax.axvspan(a, b, color=IQS_VERDE, alpha=0.13, lw=0)
ax.plot(X_PAS, 100 * CERCA_H, '-', color=GRIS, lw=1.6, label='el histograma de S18')
ax.plot(X_PAS, 100 * CERCA_PF, '-', color=IQS_AZUL, lw=2, label='200 partículas')
ax.plot(X_PAS[Z_PAS], 100 * CERCA_PF[Z_PAS], 'o', color=IQS_AZUL, ms=6, label='«hueco»')
ax.plot(X_PAS[~Z_PAS], 100 * CERCA_PF[~Z_PAS], 'o', mfc='white', mec=IQS_AZUL, ms=5, label='«pared»')
ax.set_ylim(0, 105); ax.set_ylabel('a ±0,3 m (%)'); ax.set_xlim(0, 13); ax.set_xticks(range(14))
ax.set_xlabel('x en el pasillo (m)'); ejes(ax)
ax.set_title('La creencia a ±0,3 m de la verdad, en cada medida', fontsize=10, loc='left')
ax.legend(fontsize=9, ncol=4, loc='lower center', frameon=False)
plt.show()

**Qué tiene que salir:** a ±0,3 m, del 8 % al 96 % tras la estrecha; el histograma, 97 %.

### Cuántas partículas: 100 recorridos

Localizado: al final, más del 80 % a ±0,3 m de la verdad.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# cuántas partículas se prueban [partículas]
PARTICULAS_PASILLO = [20, 50, 200, 1000]  # @param {type:"raw"}
# cuántos recorridos, cada uno con su semilla y sus medidas [recorridos]
RECORRIDOS = 100  # @param {type:"integer"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
t0 = time.time()
OK_PASILLO = {}                                         # los recorridos localizados, con cada número de partículas
for M in PARTICULAS_PASILLO:
    ok = 0
    for sem in range(int(RECORRIDOS)):
        X_, Z_ = recorre_pasillo(sem)
        P, w = mcl_pasillo(Z_, M, 1000 + sem)[-1]
        ok += masa_cerca(P, w, X_[-1]) > 0.8
    OK_PASILLO[M] = int(ok)
OK_HIST = 0                                             # y con el histograma de S18, con las mismas medidas
for sem in range(int(RECORRIDOS)):
    X_, Z_ = recorre_pasillo(sem)
    OK_HIST += int(histograma_pasillo(Z_)[-1][np.abs(XC - X_[-1]) <= 0.3 + 1e-9].sum() > 0.8)
T_PASILLO = time.time() - t0

In [ ]:
# @title ✔ Resultado
for M in PARTICULAS_PASILLO:
    print(f'M = {M:4d}: {OK_PASILLO[M]} de {RECORRIDOS}')
print(f'el histograma de S18: {OK_HIST} de {RECORRIDOS} ({coma(T_PASILLO, 1)} s)')

fig, ax = plt.subplots(figsize=(7.5, 3.3))
nombres_ = [str(M) for M in PARTICULAS_PASILLO] + ['histograma\nde S18']
valores_ = [OK_PASILLO[M] for M in PARTICULAS_PASILLO] + [OK_HIST]
ax.bar(range(len(valores_)), valores_, width=0.6, color=[IQS_AZUL] * len(PARTICULAS_PASILLO) + [GRIS])
for i, v in enumerate(valores_):
    ax.text(i, v + 2, str(v), ha='center', fontsize=10)
ax.set_xticks(range(len(valores_)), nombres_); ax.set_xlabel('partículas'); ax.grid(axis='x')
ax.set_ylim(0, 100); ax.set_ylabel(f'localizados, de {RECORRIDOS}')
ax.set_title('Recorridos localizados: más del 80 % a ±0,3 m de la verdad, al final', fontsize=10.5)
plt.show()

**Qué tiene que salir:** 39, 54, 64 y 66 de 100; el histograma, 68.

## 2. El LiDAR y el campo de verosimilitud

*En clase, sin deteneros · diapositiva 87*

Un haz vale más cuanto más cerca de una pared acaba. El peso suma los logaritmos de 18 haces.

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
POSE_54 = VERDAD[54]
R_54 = barrido(POSE_54, np.random.default_rng(0))      # el barrido desde la pose verdadera
# la verdadera, desplazada 30 cm, girada 0,2 rad (11°) y en otro sitio libre
POSES_4 = np.array([POSE_54, POSE_54 + [0.3, 0, 0], POSE_54 + [0, 0, 0.2], [10.5, 2.5, 0.0]])
LW_4 = log_verosimilitud(POSES_4, R_54[HACES])           # el logaritmo del peso de cada una
# para la figura: el campo (lo que vale un haz según dónde acaba: la campana de 0,3 m y el fondo, como en
# log_verosimilitud) y los puntos finales de los 18 haces desde cada pose
CAMPO = 0.5 * np.exp(-0.5 * (DIST / 0.3) ** 2) / (np.sqrt(2 * np.pi) * 0.3) + 0.5 / RMAX
VUELVEN_18 = R_54[HACES] < RMAX - 1e-6
FINES_4 = [p_[:2] + R_54[HACES][VUELVEN_18, None] * np.column_stack([np.cos(p_[2] + ANG[HACES][VUELVEN_18]),
                                                                     np.sin(p_[2] + ANG[HACES][VUELVEN_18])]) for p_ in POSES_4]
NOMBRES_4 = ('la verdadera', 'desplazada 30 cm', 'girada 11°', 'en otro sitio, (10,5; 2,5)')
# los límites de los cuatro paneles: que se vean todos los puntos finales, también los que caen fuera del almacén [m]
TODOS_4 = np.vstack(FINES_4)
LIM_4 = ((min(-0.8, TODOS_4[:, 0].min() - 0.4), max(13.8, TODOS_4[:, 0].max() + 0.4)),
         (min(-0.8, TODOS_4[:, 1].min() - 0.4), max(9.8, TODOS_4[:, 1].max() + 0.4)))

In [ ]:
# @title ✔ Resultado
print('log w de la pose verdadera, desplazada 30 cm, girada 11° y en otro sitio libre, (10,5; 2,5):',
      '; '.join(coma(v, 1) for v in LW_4))

fig, axs = plt.subplots(2, 2, figsize=(12.5, 6.9), gridspec_kw=dict(hspace=0.15, wspace=0.04))
for ax, p_, fin_, nom_, lw_ in zip(axs.ravel(), POSES_4, FINES_4, NOMBRES_4, LW_4):
    im = ax.imshow(CAMPO, origin='lower', extent=(0, 13, 0, 9), cmap='Greys', interpolation='nearest')
    ax.set_xlim(*LIM_4[0]); ax.set_ylim(*LIM_4[1]); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for q in fin_:
        ax.plot([p_[0], q[0]], [p_[1], q[1]], color=AMBAR, lw=0.8, alpha=0.7)
    ax.plot(fin_[:, 0], fin_[:, 1], 'o', color=AMBAR, mec='k', mew=0.6, ms=6)
    dibuja_robot(ax, p_, color=IQS_AZUL)
    ax.set_title(f'{nom_}: log w = {coma(lw_, 1)}', fontsize=11)
fig.colorbar(im, ax=axs, fraction=0.03, pad=0.02, format=COMA, label='lo que vale un haz que acaba ahí (1/m)')
axs[0, 0].plot([], [], 'o', color=AMBAR, mec='k', label='el punto final de cada haz')
axs[0, 0].plot([], [], 'o', mfc='white', mec=IQS_AZUL, mew=2, ms=9, label='la partícula que se pesa')
axs[0, 0].legend(loc='lower left', bbox_to_anchor=(0, 1.07), ncol=2, fontsize=9, frameon=False)
plt.show()

**Qué tiene que salir:** log w = −6,5; −9,0; −21,1 y −29,3.

## 3. La nube en el almacén

*En clase, sin deteneros · diapositiva 87*

Diez mil partículas al azar; cada medio metro se mueven, se pesan con el LiDAR y se remuestrean.

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
t0 = time.time()
NUBE_LIDAR = mcl_almacen(54, 10, 10000, 222)           # 10 actualizaciones, cada 0,5 m, desde el paso 54
MS_ACT = (time.time() - t0) / 11 * 1000                 # milisegundos por actualización
# para la figura: cuatro actualizaciones, 3000 partículas de cada una (cada muestra, con su generador), la parte de
# la nube a menos de 1 m del robot y el error de la estimación
INST_NUBE = (0, 1, 3, 7)
MUESTRA_NUBE = {u: np.random.default_rng(u).choice(len(NUBE_LIDAR[u]['P']), 3000, replace=False) for u in INST_NUBE}
CERCA_1M = [np.mean(np.hypot(e['P'][:, 0] - e['verdad'][0], e['P'][:, 1] - e['verdad'][1]) < 1) for e in NUBE_LIDAR]
ERR_NUBE = np.array([e['err'] for e in NUBE_LIDAR])

In [ ]:
# @title ✔ Resultado
print(f'{coma(MS_ACT, 0)} ms por actualización')
for u in (0, 3, 7):
    e = NUBE_LIDAR[u]
    print(f"actualización {u}: estimación ({coma(e['est'][0])}; {coma(e['est'][1])}; {coma(e['est'][2]*180/np.pi, 1)}°), a {coma(e['err']*100, 1)} cm")

fig = plt.figure(figsize=(15, 6.0))
gs = fig.add_gridspec(2, len(INST_NUBE), height_ratios=[1.15, 1], hspace=0.22, wspace=0.04)
for j, u in enumerate(INST_NUBE):
    ax, e = fig.add_subplot(gs[0, j]), NUBE_LIDAR[u]
    dibuja_almacen(ax)
    ax.scatter(e['P'][MUESTRA_NUBE[u], 0], e['P'][MUESTRA_NUBE[u], 1], s=2.5, color=IQS_AZUL, alpha=0.45, lw=0)
    dibuja_robot(ax, e['verdad'])
    ax.plot(*e['est'][:2], 'x', color='white', mew=3.6, ms=8, zorder=8)        # la estimación: una x fina con borde
    ax.plot(*e['est'][:2], 'x', color=IQS_AZUL, mew=1.8, ms=7, zorder=8)       # blanco, que no tapa la nube
    ax.set_title(f'actualización {u}: {coma(100*CERCA_1M[u], 0)} % a menos de 1 m', fontsize=10)
ax = fig.add_subplot(gs[1, :])
ax.semilogy(range(len(ERR_NUBE)), ERR_NUBE, 'o-', color=ROJO, lw=2, ms=5, label='el error de la estimación')
ax.axhline(0.3, color='k', ls='--', lw=1, label='30 cm')
ax.set_xlabel('actualización (cada 0,5 m)'); ax.set_ylabel('error (m)'); ax.set_xticks(range(len(ERR_NUBE))); ejes(ax)
ax.yaxis.set_minor_locator(plt.NullLocator())
ax.legend(handles=ax.get_legend_handles_labels()[0] + [
    Line2D([], [], color=IQS_AZUL, marker='.', ls='none', ms=8, label='la nube (3000 de 10 000)'),
    Line2D([], [], color=IQS_AZUL, marker='x', mew=1.8, ls='none', ms=8, label='la estimación: la media pesada'),
    Line2D([], [], color='k', marker='o', mfc='white', ls='none', ms=9, label='el robot')], fontsize=9, ncol=5,
    loc='upper center', bbox_to_anchor=(0.5, -0.3), frameon=False)
plt.show()

**Qué tiene que salir:** a menos de 1 m, 4, 12, 63 y 100 %; en la actualización 7, 4,1 cm.

### El rumbo medio

Cerca de ±180°, la media de los números no vale; la circular sí.

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
OESTE = mcl_almacen(372, 8, 10000, 226)[-1]            # la nube tras 8 actualizaciones, hacia el oeste
RUMBO_NUMEROS = np.sum(OESTE['w'] * OESTE['P'][:, 2])  # la media pesada de los números [rad]
RUMBO_CIRCULAR = media_circular(OESTE['P'][:, 2], OESTE['w'])   # la media circular, atan2(Σ w·sen θ, Σ w·cos θ) [rad]
# para la figura: los rumbos de la nube, pesados, en sectores de 2° (en una recta) y de 5° (en el círculo) [°]
RUMBOS_H, RUMBOS_B = np.histogram(np.degrees(OESTE['P'][:, 2]), bins=180, range=(-180, 180), weights=OESTE['w'])
RUMBOS_H5, RUMBOS_B5 = np.histogram(np.degrees(OESTE['P'][:, 2]), bins=72, range=(-180, 180), weights=OESTE['w'])

In [ ]:
# @title ✔ Resultado
print('hacia el oeste: media de los números', coma(RUMBO_NUMEROS*180/np.pi, 1), '° ; media circular',
      coma(RUMBO_CIRCULAR*180/np.pi, 1), '°')

fig = plt.figure(figsize=(12, 4.6))
gs = fig.add_gridspec(1, 2, width_ratios=[1.5, 1], wspace=0.25)
a1, a2 = fig.add_subplot(gs[0]), fig.add_subplot(gs[1], projection='polar')
a1.stairs(RUMBOS_H, RUMBOS_B, fill=True, color=IQS_AZUL, alpha=0.6, zorder=3)
a1.axvline(RUMBO_NUMEROS * 180 / np.pi, color=ROJO, lw=2.2)
a1.text(RUMBO_NUMEROS * 180 / np.pi - 3, RUMBOS_H.max() * 1.02, f'{coma(RUMBO_NUMEROS*180/np.pi, 1)}°', ha='right',
        va='top', fontsize=10, color=ROJO)
a1.plot(RUMBO_CIRCULAR * 180 / np.pi, RUMBOS_H.max() * 1.1, 'v', color=IQS_AZUL, ms=11, clip_on=False, zorder=4)
a1.text(RUMBO_CIRCULAR * 180 / np.pi - 6, RUMBOS_H.max() * 1.02, f'{coma(RUMBO_CIRCULAR*180/np.pi, 1)}°', ha='right',
        va='top', fontsize=10, color=IQS_AZUL, bbox=dict(fc='white', ec='none', pad=1))
a1.set_xlim(-190, 190); a1.set_xticks(range(-180, 181, 45)); a1.set_xlabel('rumbo de la partícula (°)')
a1.set_ylabel('peso'); a1.set_ylim(0, RUMBOS_H.max() * 1.1); ejes(a1)
a1.set_title('Los números: dos grupos, en los dos extremos', fontsize=11, pad=14)
a2.bar(np.radians(RUMBOS_B5[:-1] + 2.5), RUMBOS_H5, width=np.radians(5), color=IQS_AZUL, alpha=0.35)
r_ = RUMBOS_H5.max()                                    # las flechas, más largas que la barra más alta y encima
a2.annotate('', (RUMBO_NUMEROS, r_ * 1.2), (0, 0), arrowprops=dict(arrowstyle='-|>', color=ROJO, lw=2.2), zorder=5)
a2.annotate('', (RUMBO_CIRCULAR, r_ * 1.2), (0, 0), arrowprops=dict(arrowstyle='-|>', color=IQS_AZUL, lw=2.2), zorder=5)
a2.plot([OESTE['verdad'][2]] * 2, [0, r_ * 1.3], color='k', ls='--', lw=1.2)
a2.set_rlim(0, r_ * 1.3); a2.set_yticklabels([]); a2.tick_params(labelsize=9)
a2.set_xticks(np.radians(range(0, 360, 45)), [f'{g}°'.replace('-', '−') for g in (0, 45, 90, 135, 180, -135, -90, -45)])
a2.set_title('En el círculo: un solo grupo, hacia el oeste', fontsize=11, pad=24)
fig.legend(handles=[Patch(color=IQS_AZUL, alpha=0.6, label='los rumbos de la nube'),
                    Line2D([], [], color=ROJO, lw=2.2, label=f'la media de los números: {coma(RUMBO_NUMEROS*180/np.pi, 1)}°'),
                    Line2D([], [], color=IQS_AZUL, lw=2.2, label=f'la media circular: {coma(RUMBO_CIRCULAR*180/np.pi, 1)}°'),
                    Line2D([], [], color='k', ls='--', lw=1.2, label='el rumbo del robot: ±180°')],
           loc='lower center', bbox_to_anchor=(0.5, -0.06), fontsize=9, ncol=4, frameon=False)
plt.show()

**Qué tiene que salir:** 92,8° frente a 178,6°.

## 4. Taller: cuántas hacen falta

*En clase · diapositiva 88*

Arranques al azar en la ruta, sin saber dónde. Localizado: a los 8 m, a menos de 30 cm y 5° del robot.

### En clase: veinte arranques con 3000 y con 5000

Ejecutad (unos 5 s) y comparad con la tabla de la 88 (50 arranques).

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# cuántas partículas [partículas]
PARTICULAS = [3000, 5000]  # @param {type:"raw"}
# cuántos arranques: los primeros de los 50 de la tabla de la diapositiva 88 [de 1 a 50]
ARRANQUES = 20  # @param {type:"integer"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
ARR = np.random.default_rng(11).integers(0, len(VERDAD) - 170, 50)[:int(ARRANQUES)]   # el paso de cada arranque
FINALES, EXITOS, SEGUNDOS = {}, {}, {}                  # con cada M: la nube a los 8 m, si se localiza, y lo que tarda
for M in PARTICULAS:
    t0 = time.time()
    FINALES[M] = [mcl_almacen(int(k0), 16, M, 200 + i)[-1] for i, k0 in enumerate(ARR)]
    EXITOS[M] = [localizado(e) for e in FINALES[M]]
    SEGUNDOS[M] = time.time() - t0
TABLA_88 = {1000: 18, 3000: 30, 10000: 47}              # se localizan, de 50 arranques: la tabla de la diapositiva 88
# para la figura: la tasa de cada uno y su incertidumbre, una σ = √(p·(1 − p)/N), y el error final de cada arranque
TASA_88 = {m: (k / 50, np.sqrt(k / 50 * (1 - k / 50) / 50)) for m, k in TABLA_88.items()}
TASA_N = {m: (np.mean(EXITOS[m]), np.sqrt(np.mean(EXITOS[m]) * (1 - np.mean(EXITOS[m])) / len(ARR))) for m in PARTICULAS}
M_TODAS = sorted({*TABLA_88, *PARTICULAS})
ERR_FINAL = {m: np.array([e['err'] for e in FINALES[m]]) for m in PARTICULAS}

In [ ]:
# @title ✔ Resultado
for M in PARTICULAS:
    print(f'M = {M:5d}: se localizan {sum(EXITOS[M])} de {len(ARR)} ({coma(SEGUNDOS[M], 1)} s)')

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.4), gridspec_kw=dict(width_ratios=[1, 1.35], wspace=0.22))
for i, m in enumerate(M_TODAS):
    for dx, tasas_, n_, color in ((-0.2, TASA_88, 50, GRIS), (0.2, TASA_N, len(ARR), IQS_AZUL)):
        if m in tasas_:
            p_, s_ = tasas_[m]
            dx = dx if m in TASA_88 and m in TASA_N else 0     # en pareja solo si los dos tienen ese M
            a1.bar(i + dx, 100 * p_, width=0.38, color=color)
            a1.errorbar(i + dx, 100 * p_, yerr=100 * s_, color='k', capsize=4, lw=1.2)
            a1.text(i + dx, 100 * (p_ + s_) + 2, f'{round(p_ * n_)}\nde {n_}', ha='center', va='bottom', fontsize=9,
                    linespacing=1.0)                    # en dos líneas: no se pisan las de una pareja
a1.set_xticks(range(len(M_TODAS)), [f'{m:,}'.replace(',', ' ') if m >= 10000 else str(m) for m in M_TODAS])
a1.set_xlabel('partículas'); a1.set_ylabel('se localizan (%)'); a1.set_ylim(0, 118); a1.grid(axis='x')
a1.legend(handles=[Patch(color=GRIS, label='la tabla de la 88: 50 arranques'),
                   Patch(color=IQS_AZUL, label=f'vuestros: {len(ARR)} arranques'),
                   Line2D([], [], color='k', marker='_', ms=10, lw=1.2, label='±1σ = √(p·(1 − p)/N)')],
          fontsize=9, loc='upper left', frameon=True, framealpha=1, edgecolor='none')
a1.set_title('La tasa de éxito y su incertidumbre', fontsize=11)
MARCAS = 'osD^v'
for j, m in enumerate(PARTICULAS):
    x_ = np.arange(1, len(ARR) + 1) + (j - (len(PARTICULAS) - 1) / 2) * 0.18
    a2.scatter(x_, ERR_FINAL[m], s=42, marker=MARCAS[j % 5], zorder=3,
               c=[IQS_VERDE if ok_ else ROJO for ok_ in EXITOS[m]], edgecolors='k', linewidths=0.5)
    a2.plot([], [], MARCAS[j % 5], color='w', mec='k', ms=7, label=f'M = {m}')
a2.axhline(0.3, color='k', ls='--', lw=1)
a2.text(len(ARR) + 0.6, 0.3, '30 cm', va='center', fontsize=9)
a2.set_yscale('log'); a2.yaxis.set_major_formatter(COMA); a2.yaxis.set_minor_locator(plt.NullLocator())
a2.set_xticks(range(1, len(ARR) + 1) if len(ARR) <= 25 else [1, *range(5, len(ARR) + 1, 5)]); a2.set_xlim(0.4, len(ARR) + 0.6)
a2.set_xlabel('arranque'); a2.set_ylabel('error final, a los 8 m (m)')
a2.legend(handles=a2.get_legend_handles_labels()[0] + [Patch(color=IQS_VERDE, label='localizado'), Patch(color=ROJO, label='no')],
          fontsize=9, ncol=4, loc='upper center', bbox_to_anchor=(0.5, -0.13), frameon=False)
a2.set_title('El error final de cada arranque', fontsize=11)
plt.show()

**Qué tiene que salir:** 8 y 15 de 20 con 3000 y 5000; la tabla, 30 de 50 con 3000.

## 5. Taller: el EKF que arranca en el hueco equivocado

*En clase · diapositiva 91*

El robot, frente a la puerta estrecha; el EKF de S18, frente a la ancha. Cada medida va a la baliza que mejor cuadra.

### En clase: el EKF y la nube, con las mismas balizas

Ejecutad: ¿dónde acaba el EKF, cuánto se fía de sí mismo, y la nube?

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# el robot arranca en este paso de la ruta (266: en el pasillo, frente a la puerta estrecha)
K0 = 266  # @param {type:"integer"}
# dónde cree estar el EKF al arrancar: x [m], y [m] y rumbo [rad]; (1.9, 5.2, 0.0), frente a la puerta ancha
EKF_INICIO = (1.9, 5.2, 0.0)  # @param {type:"raw"}
# su σ al arrancar: a lo largo [m], de lado [m] y del rumbo [°]
EKF_SIGMA = (0.5, 0.05, 2)  # @param {type:"raw"}
# la puerta de Mahalanobis de S18 [χ² con 2 grados de libertad: el 99 %]
PUERTA = 9.21  # @param {type:"number"}
# la nube desde cero, con las mismas balizas: cuántas partículas y cuántas actualizaciones de 0,5 m
NUBE_M = 10000  # @param {type:"integer"}
NUBE_ACTUALIZACIONES = 10  # @param {type:"integer"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
odo5, med5 = simula_sin_nombre(3, int(K0))             # lo que leen los sensores desde K0: balizas sin nombre
VERDAD_5 = VERDAD[int(K0):]                             # el robot, desde K0
SIGMA0_5 = np.diag([EKF_SIGMA[0]**2, EKF_SIGMA[1]**2, (EKF_SIGMA[2]*D2R)**2])
EKF_5 = []                                              # sin puerta y con ella: lo que cree el EKF, su error y su σ
for puerta in (None, PUERTA):
    Xe, Pe = ekf_vecino(odo5, med5, EKF_INICIO, SIGMA0_5, k0=int(K0), puerta=puerta)
    err = np.hypot(*(Xe[-1, :2] - VERDAD_5[-1, :2]))
    sigma = np.sqrt(np.linalg.eigvalsh(Pe[-1][:2, :2]).max())
    EKF_5.append(dict(puerta=puerta, X=Xe, err=err, sigma=sigma,          # al final, y paso a paso para la figura
                      errs=np.hypot(Xe[:, 0] - VERDAD_5[:, 0], Xe[:, 1] - VERDAD_5[:, 1]),
                      sigmas=np.sqrt([np.linalg.eigvalsh(p_[:2, :2]).max() for p_ in Pe])))
NUBE_BALIZAS = mcl_almacen(int(K0), int(NUBE_ACTUALIZACIONES), int(NUBE_M), 30, sensor='balizas')
# para la figura: el recorrido desde el arranque [m], la estimación de la nube y su error
D_EKF = np.arange(len(VERDAD_5)) * DD
D_NUBE = np.array([(e['k'] - int(K0)) * DD for e in NUBE_BALIZAS])
EST_NUBE = np.array([e['est'] for e in NUBE_BALIZAS])
ERR_NUBE_5 = np.array([e['err'] for e in NUBE_BALIZAS])
# dónde acaba cada uno, y los límites del panel: que se vean las trayectorias enteras, aunque salgan del almacén [m]
FIN_5 = [r_['X'][-1, :2] for r_ in EKF_5]
FIN_ROBOT_5 = VERDAD_5[-1, :2]                          # el robot al acabar el EKF (la ruta entera)
FIN_ROBOT_NUBE = VERDAD[NUBE_BALIZAS[-1]['k']][:2]      # y al acabar la nube (tras NUBE_ACTUALIZACIONES)
TODO_5 = np.vstack([VERDAD_5[:, :2], EST_NUBE[:, :2]] + [r_['X'][:, :2] for r_ in EKF_5])
LIM_5 = ((min(-0.4, TODO_5[:, 0].min() - 0.6), max(13.4, TODO_5[:, 0].max() + 0.6)),
         (min(-0.4, TODO_5[:, 1].min() - 0.6), max(9.4, TODO_5[:, 1].max() + 0.6)))

In [ ]:
# @title ✔ Resultado
for r_ in EKF_5:
    nombre_ = 'sin puerta' if r_['puerta'] is None else 'con la puerta de ' + coma(r_['puerta'])
    print(f"{nombre_}: acaba a {coma(r_['err'])} m del robot con σ = {coma(r_['sigma']*100, 1)} cm ({coma(r_['err']/r_['sigma'], 0)} σ)")
print(f"MCL con las mismas balizas sin nombre, desde cero: en la actualización 1, a {coma(NUBE_BALIZAS[1]['err']*100, 1)} cm ;"
      f" al final, a {coma(NUBE_BALIZAS[-1]['err']*100, 1)} cm")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 5.4), gridspec_kw=dict(width_ratios=[1.9, 1], wspace=0.15))
COLORES_5 = (IQS_AZUL, AMBAR)                           # el EKF sin puerta, en azul como en la 91; con ella, en ámbar
dibuja_almacen(a1, metros=True)
a1.set_xlim(*LIM_5[0]); a1.set_ylim(*LIM_5[1]); a1.xaxis.set_major_locator(plt.MultipleLocator(2))
a1.plot(BALIZAS[:, 0], BALIZAS[:, 1], '*', color=IQS_VERDE, ms=12, mec='white', mew=0.6, zorder=5)
a1.plot(VERDAD_5[:, 0], VERDAD_5[:, 1], color='k', lw=2.6, zorder=8, label='el robot')   # encima de la nube
for r_, fin_, color in zip(EKF_5, FIN_5, COLORES_5):
    a1.plot(r_['X'][:, 0], r_['X'][:, 1], color=color, lw=1.8)
    a1.plot(*fin_, 'X', color=color, mec='white', mew=1.2, ms=13, zorder=9)
    a1.annotate(f"{coma(r_['err'])} m, σ = {coma(r_['sigma']*100, 1)} cm", fin_, (fin_[0] + 0.35, fin_[1] - 0.75), fontsize=9.5,
                color=color, fontweight='bold', bbox=dict(fc='white', ec='none', alpha=0.85, pad=1), zorder=10)
a1.plot(EST_NUBE[:, 0], EST_NUBE[:, 1], 'o-', color=IQS_VERDE, lw=1.6, ms=5.5, zorder=7)
a1.plot(*EST_NUBE[-1, :2], 'X', color=IQS_VERDE, mec='white', mew=1.2, ms=16, zorder=9)   # más grande: el robot, encima
for fin_ in (FIN_ROBOT_5, FIN_ROBOT_NUBE):              # el robot cuando acaba el EKF y cuando acaba la nube
    a1.plot(*fin_, 'o', color='k', mec='white', ms=10, zorder=9)
a1.annotate(f"{coma(ERR_NUBE_5[1]*100, 1)} cm en la 1;\n{coma(ERR_NUBE_5[-1]*100, 1)} cm a los {coma(D_NUBE[-1], 0)} m",
            EST_NUBE[-1, :2], (EST_NUBE[-1, 0] + 0.3, EST_NUBE[-1, 1] + 0.45), fontsize=9, color=IQS_VERDE,
            fontweight='bold', bbox=dict(fc='white', ec='none', alpha=0.85, pad=1), zorder=10)
a1.plot(*EKF_INICIO[:2], 'o', mfc='white', mec=GRIS, mew=2, ms=9, zorder=8)
a1.annotate('el EKF arranca aquí', EKF_INICIO[:2], (-0.5, EKF_INICIO[1] + 0.8), ha='right', va='center', fontsize=9,
            arrowprops=dict(arrowstyle='-', color=GRIS), bbox=dict(fc='white', ec='none', alpha=0.85, pad=1))
dibuja_robot(a1, VERDAD_5[0])
a1.set_title('Por dónde va cada uno, y a qué distancia del robot acaba', fontsize=11)
for r_, color in zip(EKF_5, COLORES_5):
    nombre_ = 'sin puerta' if r_['puerta'] is None else 'con la puerta'
    a2.plot(D_EKF, r_['errs'], color=color, lw=2, label=f'el EKF {nombre_}: su error')
    a2.plot(D_EKF, r_['sigmas'], color=color, lw=1.6, ls=':', label='y su σ')
a2.plot(D_NUBE, ERR_NUBE_5, 'o-', color=IQS_VERDE, lw=2, ms=5, label='la nube desde cero: su error')
a2.set_yscale('log'); a2.yaxis.set_major_formatter(COMA); a2.yaxis.set_minor_locator(plt.NullLocator())
a2.xaxis.set_major_formatter(COMA)
a2.set_xlabel('recorrido desde el arranque (m)'); a2.set_ylabel('error y σ (m)')
a2.set_title('El error, y lo que cree el EKF de sí mismo', fontsize=11)
h_ = a2.get_legend_handles_labels()[0]                 # en filas: cada error junto a su σ
a2.legend(handles=[h_[0], h_[2], h_[4], h_[1], h_[3]], fontsize=9, ncol=2, loc='upper center', bbox_to_anchor=(0.5, -0.13),
          frameon=False)
a1.legend(handles=[Line2D([], [], color='k', lw=2.6, label='el robot'),
                   Line2D([], [], color=IQS_AZUL, lw=1.8, label='el EKF sin puerta'),
                   Line2D([], [], color=AMBAR, lw=1.8, label='el EKF con la puerta'),
                   Line2D([], [], color=IQS_VERDE, marker='o', ms=4, lw=1.6, label='la nube desde cero'),
                   Line2D([], [], color='k', marker='o', mec='white', ms=9, ls='none',
                          label='dónde está el robot cuando acaba cada una'),
                   Line2D([], [], color=GRIS, marker='X', mec='white', ms=11, ls='none', label='dónde acaba cada estimación'),
                   Line2D([], [], color=IQS_VERDE, marker='*', ms=11, ls='none', label='las balizas')],
          fontsize=9, ncol=4, loc='upper center', bbox_to_anchor=(0.5, -0.11), frameon=False)
plt.show()

**Qué tiene que salir:** sin puerta, 8,50 m y σ = 4,5 cm; con ella, 6,55 m y σ = 47,6 cm; la nube, 1,8 cm.

## 6. Taller: el robot secuestrado

*En clase · diapositiva 89*

Sale del muelle localizado; en la actualización 10, alguien lo lleva de la puerta ancha a la estrecha.

### En clase: sin partículas nuevas, con un 5 % y con un 2 %

Ejecutad: ¿se recupera en cada caso, y desde qué actualización?

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# la fracción de partículas nuevas al azar en cada actualización [de 0 a 1]: ninguna, un 5 % y un 2 %
INYECCIONES = [0.0, 0.05, 0.02]  # @param {type:"raw"}
# el secuestro: en qué actualización [de 1 a 20; cada 0,5 m]
ACT_SECUESTRO = 10  # @param {type:"integer"}
# y en qué paso de la ruta aparece el robot [de 0 a 544] (330: bajando de la estrecha)
PASO_DESTINO = 330  # @param {type:"integer"}
# la semilla (400: la de la figura de la diapositiva 89)
SEMILLA_SECUESTRO = 400  # @param {type:"integer"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
P0 = alrededor(VERDAD[0], 10000, np.random.default_rng(0))   # 10 000 partículas en el muelle: localizado
S6 = int(ACT_SECUESTRO)
SECUESTROS = []                                         # con cada inyección: las 31 actualizaciones y desde cuándo se localiza
for iny in INYECCIONES:
    r = mcl_almacen(0, 30, 10000, int(SEMILLA_SECUESTRO), P0=P0, secuestro=(S6, int(PASO_DESTINO)), inyecta=iny)
    ok = [localizado(e) for e in r]
    desde = next((u for u in range(S6, len(ok)) if all(ok[u:])), None)
    SECUESTROS.append(dict(iny=iny, r=r, desde=desde, errs=np.array([e['err'] for e in r])))
# para la figura: el color de cada caso (como en la 89), la nube 2 m después del secuestro (1500 partículas, con su
# generador) y dónde estaba el robot cuando se lo llevaron
COLOR_INY = {0.0: ROJO, 0.05: IQS_VERDE, 0.02: AMBAR}
COLORES_6 = [COLOR_INY.get(s_['iny'], c_) for s_, c_ in zip(SECUESTROS, [IQS_AZUL, GRIS, 'k'] * 3)]
U_VISTA = min(S6 + 4, 30)
U_DESPUES = [min(S6 + d, 30) for d in (1, 3, 10)]      # 1, 3 y 10 actualizaciones después (como mucho, la 30)
MUESTRA_6 = np.random.default_rng(1).choice(10000, 1500, replace=False)
ANTES_6 = VERDAD[min(S6 * CADA, len(U))]
PESO_1M_6 = []                                          # en esa actualización: el peso de la nube a menos de 1 m del robot
for s_ in SECUESTROS:
    e_ = s_['r'][U_VISTA]
    PESO_1M_6.append(float(np.sum(e_['w'][np.hypot(e_['P'][:, 0] - e_['verdad'][0], e_['P'][:, 1] - e_['verdad'][1]) < 1])))

In [ ]:
# @title ✔ Resultado
for s_ in SECUESTROS:
    print(f"inyección {coma(100*s_['iny'], 0)} %: error 1, 3 y 10 actualizaciones después:",
          ', '.join(coma(s_['r'][u]['err'], 2) for u in U_DESPUES), 'm ;',
          'no se recupera' if s_['desde'] is None else f"localizada desde la actualización {s_['desde']}")

fig = plt.figure(figsize=(14, 7.2))
gs = fig.add_gridspec(2, len(SECUESTROS), height_ratios=[1, 1.05], hspace=0.34, wspace=0.05, bottom=0.08)
a0 = fig.add_subplot(gs[0, :])
for s_, color in zip(SECUESTROS, COLORES_6):
    a0.semilogy(range(len(s_['errs'])), s_['errs'], '-o', ms=3.5, lw=2, color=color, label=f"{coma(100*s_['iny'], 0)} % nuevas")
a0.axvline(S6, color='k', ls=':', lw=1.5)
a0.text(S6 + 0.2, 0.012, 'secuestro', fontsize=9.5)
a0.axhline(0.3, color='k', ls='--', lw=1)
a0.text(31.4, 0.32, '30 cm', ha='right', va='bottom', fontsize=9)
a0.set_xlim(-0.5, 31.5); a0.set_xticks(range(0, 31, 2)); a0.yaxis.set_major_formatter(COMA)
a0.yaxis.set_minor_locator(plt.NullLocator())
a0.set_xlabel('actualización (cada 0,5 m)'); a0.set_ylabel('error de la estimación (m)')
a0.legend(fontsize=9, ncol=len(SECUESTROS), loc='upper left', frameon=True, framealpha=0.9)
a0.set_title('El error en cada actualización, y la nube 2 m después del secuestro', fontsize=11)
for j, (s_, color) in enumerate(zip(SECUESTROS, COLORES_6)):
    ax, e = fig.add_subplot(gs[1, j]), s_['r'][U_VISTA]
    dibuja_almacen(ax)
    ax.scatter(e['P'][MUESTRA_6, 0], e['P'][MUESTRA_6, 1], s=2, color=IQS_AZUL, alpha=0.45, lw=0)
    dibuja_robot(ax, ANTES_6, color=GRIS)
    dibuja_robot(ax, e['verdad'])
    ax.plot(*e['est'][:2], 'x', color='white', mew=3.6, ms=8, zorder=8)        # la estimación: una x fina con borde
    ax.plot(*e['est'][:2], 'x', color=IQS_AZUL, mew=1.8, ms=7, zorder=8)       # blanco, que no tapa la nube
    for sp in ax.spines.values():
        sp.set_color(color); sp.set_linewidth(3)
    ax.set_title(f"{coma(100*s_['iny'], 0)} % nuevas: la estimación, a {coma(e['err'], 1)} m del robot\n"
                 f"el {coma(100*PESO_1M_6[j], 0)} % del peso, a menos de 1 m", fontsize=10)
fig.legend(handles=[Line2D([], [], color=IQS_AZUL, marker='.', ls='none', ms=8, label='la nube (1500 de 10 000)'),
                    Line2D([], [], color=IQS_AZUL, marker='x', mew=1.8, ls='none', ms=8, label='la estimación: la media pesada'),
                    Line2D([], [], color='k', marker='o', mfc='white', ls='none', ms=9, label='el robot'),
                    Line2D([], [], color=GRIS, marker='o', mfc='white', ls='none', ms=9, label='donde estaba al llevárselo')],
           fontsize=9, ncol=4, loc='lower center', bbox_to_anchor=(0.5, 0.0), frameon=False)
plt.show()

**Qué tiene que salir:** sin nuevas, no se recupera; con el 5 %, desde la actualización 11; con el 2 %, desde la 15.

### Para la puesta en común

*Con cifras: se ponen en común al acabar.*

1. Cuántas hacen falta: con 3000 y 5000 en 20 arranques, ¿cuánto se mueve la tasa frente a la tabla de la 88 (50)?
2. El hueco equivocado: qué hace el EKF de S18 que arranca en la puerta ancha con el robot en la estrecha, y qué hace la nube desde cero con las mismas balizas.
3. El secuestro: sin partículas nuevas, con un 5 %, y si basta un 2 %.

### Ejercicio 1 (en casa): el detector de la puerta en el pasillo

Con el detector de 0,6 y 0,2, ¿cuántos de los 100 recorridos se localizan? ¿Lo arreglan las partículas?

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# el detector: p(«hueco» | hueco) y p(«hueco» | pared) [de 0 a 1]; el de la sección 1 es (0.8, 0.1)
E1_DETECTOR = (0.6, 0.2)  # @param {type:"raw"}
# cuántas partículas [partículas] y cuántos recorridos
E1_PARTICULAS = [200, 1000]  # @param {type:"raw"}
E1_RECORRIDOS = 100  # @param {type:"integer"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
E1_REF = globals().get('OK_PASILLO', {})                # para la figura: lo de la sección 1, si se ha ejecutado
E1_REF_HIST = globals().get('OK_HIST')
e1_antes = (P_HUECO, P_FALSA)                          # el detector de la sección 1, para dejarlo como estaba
P_HUECO, P_FALSA = E1_DETECTOR                          # verosimilitud_hueco() y recorre_pasillo() usan estos dos
try:
    E1_OK = {}                                          # los recorridos localizados con cada número de partículas
    for e1_m in E1_PARTICULAS:
        e1_n = 0
        for e1_s in range(int(E1_RECORRIDOS)):
            e1_x, e1_z = recorre_pasillo(e1_s)
            e1_p, e1_w = mcl_pasillo(e1_z, e1_m, 1000 + e1_s)[-1]
            e1_n += masa_cerca(e1_p, e1_w, e1_x[-1]) > 0.8
        E1_OK[e1_m] = int(e1_n)
    E1_HIST = 0                                         # y con el histograma
    for e1_s in range(int(E1_RECORRIDOS)):
        e1_x, e1_z = recorre_pasillo(e1_s)
        E1_HIST += int(histograma_pasillo(e1_z)[-1][np.abs(XC - e1_x[-1]) <= 0.3 + 1e-9].sum() > 0.8)
finally:
    P_HUECO, P_FALSA = e1_antes

In [ ]:
# @title ✔ Resultado
print(f'detector {coma(E1_DETECTOR[0], 1)} / {coma(E1_DETECTOR[1], 1)}: ' +
      ' ; '.join(f'M = {m}: {E1_OK[m]} de {E1_RECORRIDOS}' for m in E1_PARTICULAS) + f' ; el histograma: {E1_HIST} de {E1_RECORRIDOS}')

fig, ax = plt.subplots(figsize=(7.5, 3.3))
e1_nombres = [f'M = {m}' for m in E1_PARTICULAS] + ['histograma']
e1_seccion1 = [E1_REF.get(m) for m in E1_PARTICULAS] + [E1_REF_HIST]
e1_vuestro = [E1_OK[m] for m in E1_PARTICULAS] + [E1_HIST]
for i, (a_, b_) in enumerate(zip(e1_seccion1, e1_vuestro)):
    dx_ = 0.2 if a_ is not None else 0                  # en pareja solo si hay con qué compararlo
    if a_ is not None:
        ax.bar(i - dx_, a_, width=0.38, color=GRIS); ax.text(i - dx_, a_ + 2, str(a_), ha='center', fontsize=10)
    ax.bar(i + dx_, b_, width=0.38, color=IQS_AZUL); ax.text(i + dx_, b_ + 2, str(b_), ha='center', fontsize=10)
ax.set_xticks(range(len(e1_nombres)), e1_nombres); ax.grid(axis='x'); ax.set_ylim(0, 112); ax.set_yticks(range(0, 101, 20))
ax.set_ylabel(f'localizados, de {E1_RECORRIDOS}')
ax.legend(handles=[Patch(color=GRIS, label='el detector de la sección 1: 0,8 y 0,1')] * any(a_ is not None for a_ in e1_seccion1) +
                  [Patch(color=IQS_AZUL, label=f'el vuestro: {coma(E1_DETECTOR[0], 1)} y {coma(E1_DETECTOR[1], 1)}')],
          fontsize=9, loc='upper right', frameon=False)
ax.set_title(f'Recorridos localizados, de {E1_RECORRIDOS}, con cada detector', fontsize=11)
plt.show()

### Ejercicio 2 (en casa): pocas partículas y muchas nuevas

Con los 20 arranques de la sección 4, ¿compensa un 10 % de partículas nuevas tener solo 1000 o 300?

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# los casos: (partículas, fracción de partículas nuevas al azar en cada actualización [de 0 a 1])
E2_CASOS = [(1000, 0.0), (1000, 0.10), (300, 0.0), (300, 0.10)]  # @param {type:"raw"}
# cuántos arranques: los primeros de la tabla de la diapositiva 88, como en la sección 4 [de 1 a 50]
E2_ARRANQUES = 20  # @param {type:"integer"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
e2_arr = np.random.default_rng(11).integers(0, len(VERDAD) - 170, 50)[:int(E2_ARRANQUES)]   # los de la sección 4
E2 = []                                                 # (partículas, inyección, cuántos se localizan)
for e2_m, e2_iny in E2_CASOS:
    E2.append((e2_m, e2_iny, sum(localizado(mcl_almacen(int(e2_k0), 16, e2_m, 200 + e2_i, inyecta=e2_iny)[-1])
                                 for e2_i, e2_k0 in enumerate(e2_arr))))

In [ ]:
# @title ✔ Resultado
for e2_m, e2_iny, e2_ok in E2:
    print(f'M = {e2_m}, un {coma(100*e2_iny, 0)} % nuevas: {e2_ok} de {len(e2_arr)}')

fig, ax = plt.subplots(figsize=(7.5, 3.3))
ax.bar(range(len(E2)), [100 * c_[2] / len(e2_arr) for c_ in E2], width=0.6,
       color=[IQS_AZUL if c_[1] == 0 else IQS_VERDE for c_ in E2])
for i, (e2_m, e2_iny, e2_ok) in enumerate(E2):
    ax.text(i, 100 * e2_ok / len(e2_arr) + 2, f'{e2_ok} de {len(e2_arr)}', ha='center', fontsize=10)
ax.set_xticks(range(len(E2)), [f'M = {c_[0]}\n{coma(100*c_[1], 0)} % nuevas' for c_ in E2]); ax.grid(axis='x')
ax.set_ylim(0, 100); ax.set_ylabel('se localizan (%)')
ax.set_title(f'Se localizan, de {len(e2_arr)} arranques', fontsize=11)
plt.show()

### Ejercicio 3 (en casa): inyectar solo cuando hace falta

Se inyecta max(0, 1 − w_rápida/w_lenta), con dos medias de la verosimilitud. En el secuestro (unos 15 s en Colab), ¿cuántas se recuperan y cuánto inyecta?

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# cuánto se acerca cada media a la verosimilitud media de la nube en cada actualización [de 0 a 1]
E3_LENTA = 0.05  # @param {type:"number"}
E3_RAPIDA = 0.5  # @param {type:"number"}
# cuántas semillas, desde la 400 de la sección 6 [semillas]
E3_SEMILLAS = 20  # @param {type:"integer"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
def mcl_adaptativa(semilla, lenta, rapida, M=10000, n=30):
    """El bucle de mcl_almacen() con el secuestro de la sección 6, pero inyectando la fracción max(0, 1 − w_rápida/w_lenta):
    w_lenta y w_rápida siguen la verosimilitud media de la nube, media(exp(log w / 18)), una despacio y otra deprisa.
    Devuelve, por actualización, si está localizada y la fracción que inyecta."""
    rng = np.random.default_rng(semilla)
    odo = U + np.random.default_rng(semilla + 1_000_000).normal(0, [SIG_D, SIG_T], size=U.shape)   # como mcl_almacen
    P = alrededor(VERDAD[0], M, np.random.default_rng(semilla - 400))
    k, w_lenta, w_rapida, fracciones, ok = 0, None, None, [], []
    for u in range(n + 1):
        if u == int(ACT_SECUESTRO):
            k = int(PASO_DESTINO)
        lw = log_verosimilitud(P, barrido(VERDAD[k], rng, HACES))
        media = np.mean(np.exp(lw / HACES.size)[np.isfinite(lw)]) if np.isfinite(lw).any() else 0.0
        w_lenta = media if w_lenta is None else w_lenta + lenta * (media - w_lenta)
        w_rapida = media if w_rapida is None else w_rapida + rapida * (media - w_rapida)
        w = pesos(lw)
        est = np.array([np.sum(w * P[:, 0]), np.sum(w * P[:, 1]), media_circular(P[:, 2], w)])
        ok.append(np.hypot(*(est[:2] - VERDAD[k][:2])) < 0.3 and abs(envuelve(est[2] - VERDAD[k][2])) < 5 * D2R)
        if u == n:
            break
        P = P[sistematico(w, rng)]
        f = max(0.0, 1 - w_rapida / w_lenta)
        m = int(round(f * M)); fracciones.append(f)
        if m:
            P[rng.choice(M, m, replace=False)] = al_azar(m, rng)
        k2 = min(k + CADA, len(U))
        d = np.zeros(3)
        for dd in odo[k:k2]:
            d = paso(d, dd)
        P = mover(P, d, rng)
        k = k2
    return ok, np.array(fracciones)


E3_TARDA, E3_ANTES, E3_DESPUES, E3_FRACCIONES = [], [], [], []   # por semilla
e3_s = int(ACT_SECUESTRO)
for e3_sem in range(400, 400 + int(E3_SEMILLAS)):
    e3_ok, e3_f = mcl_adaptativa(e3_sem, E3_LENTA, E3_RAPIDA)
    e3_u = next((u for u in range(e3_s, len(e3_ok)) if all(e3_ok[u:])), None)
    E3_TARDA.append(None if e3_u is None else e3_u - e3_s)        # cuántas actualizaciones tarda en recuperarse
    E3_ANTES.append(e3_f[:e3_s].mean()); E3_DESPUES.append(e3_f[e3_s:e3_s + 3].max())   # antes y justo después
    E3_FRACCIONES.append(e3_f)
E3_RECUPERA = [t_ for t_ in E3_TARDA if t_ is not None]
E3_MEDIANA_TARDA = np.median(E3_RECUPERA)               # cuánto tarda en recuperarse (mediana) [actualizaciones]
E3_MEDIA_ANTES = np.mean(E3_ANTES)                      # cuánto inyecta antes del secuestro (media) [de 0 a 1]
E3_MEDIANA_MAX = np.median(E3_DESPUES)                  # y justo después (mediana de los máximos) [de 0 a 1]
E3_MEDIANA_F = np.median(E3_FRACCIONES, axis=0)         # para la figura: la mediana en cada actualización

In [ ]:
# @title ✔ Resultado
print(f'se recupera en {len(E3_RECUPERA)} de {len(E3_TARDA)}, mediana {coma(E3_MEDIANA_TARDA, 1)} actualizaciones ;'
      f' antes del secuestro inyecta de media el {coma(100*E3_MEDIA_ANTES, 1)} % ;'
      f' justo después, hasta el {coma(100*E3_MEDIANA_MAX, 0)} % (mediana de los máximos)')

fig, ax = plt.subplots(figsize=(10, 3.4))
for e3_f in E3_FRACCIONES:
    ax.plot(range(len(e3_f)), 100 * e3_f, color=GRIS, lw=0.8, alpha=0.6)
ax.plot(range(len(E3_MEDIANA_F)), 100 * E3_MEDIANA_F, color=IQS_AZUL, lw=2.2,
        label=f'la mediana de las {len(E3_FRACCIONES)} semillas')
ax.plot([], [], color=GRIS, lw=0.8, label='cada semilla')
ax.axvline(e3_s, color='k', ls=':', lw=1.5); ax.text(e3_s + 0.2, 95, 'secuestro', fontsize=9.5, va='top')
ax.set_ylim(0, 100); ax.set_xlabel('actualización (cada 0,5 m)'); ax.set_ylabel('partículas nuevas (%)'); ejes(ax)
ax.legend(fontsize=9, loc='upper right', frameon=False)
ax.set_title('Cuántas partículas nuevas inyecta en cada actualización', fontsize=11)
plt.show()

---

## Soluciones

**Ejercicio 1.** 5 de 100 con 200 partículas y 2 con 1000; el histograma, 5. Las partículas no lo arreglan: el problema es el sensor, y ningún número de partículas saca información de donde no la hay.

**Ejercicio 2.** Con 1000 partículas, 7 de 20 sin nuevas y 12 con un 10 %; con 300, 8 y 9. La inyección encuentra el sitio, pero hacen falta partículas para seguirlo: por eso AMCL sube el número mientras la nube está repartida.

**Ejercicio 3.** Se recuperan las 20, en 3 actualizaciones de mediana. Antes del secuestro inyecta un 3,6 % de media; justo después, hasta un 52 %: casi nada en marcha normal, mucho cuando la nube deja de explicar lo que ve el LiDAR.

---

## Para llevarse de esta sesión

- **Una nube es una creencia con los picos que haga falta.**
- **Muestrear, pesar y remuestrear:** el filtro de Bayes de S18, con muestras.
- **Localizarse sin saber dónde pide partículas;** la tasa en N arranques se mueve unos √(p·(1 − p)/N).
- **Sin puerta, el EKF equivocado no avisa.**
- **Sin partículas nuevas, no hay recuperación de un secuestro.**

*Cuaderno del curso 82514 Mecatrónica y Robótica · IQS Universitat Ramon Llull · curso 2026/27*

*© Guillermo Reyes Carmenaty · Publicado bajo [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es): puedes usarlo, adaptarlo y redistribuirlo, incluso con fines comerciales, siempre que reconozcas la autoría.*